# Session 4: Coordinated Multiple Views

Vitessce is built around a concept called **coordinated multiple views**: every view in the grid looks up variables in the **coordination space**. The coordination space is a dictionary of named variables called *coordination types*, which is stored in the Vitessce configuration. When a user interacts with one view (selects a gene, highlights a cell cluster, zooms in), the coordination space updates and all views that are linked to this variable respond together.

In this notebook you will learn to:

1. Understand the coordination space and how views share state by default
2. Use `config.link_views_by_dict()` to set initial coordination values and explicitly link views
3. Use `config.add_coordination()` as an alternative mechanism to define how views are coordinated

In [ ]:
!pip install "vitessce[all]==3.9.2" "scanpy" "easy-vitessce==0.0.12" "spatialdata==0.7.3" "spatialdata-plot==0.4.0"

## 1. How the coordination space works

Every view in Vitessce is connected to the coordination space through **coordination scopes**, which are named variables that hold a single coordination value. Multiple views can share the same scope for a certain coordination type, which is what makes them "coordinated".


Recall the "Specifying initial visualization properties" section from the `01-Single-Spatial-View.ipynb` notebook. See the [coordinated multiple views documentation](https://vitessce.io/docs/coordination/) for more details.

## 2. Setting initial coordination values with `link_views_by_dict()`

The `link_views_by_dict()` method sets the **initial value** of one or more coordination types for a group of views. All listed views will share the same coordination scope for the listed types.

```python
config.link_views_by_dict(
    [view1, view2],
    { "featureSelection": ["CD79A"], "obsColorEncoding": "geneSelection" },
)
```

This is the most common way to:
- Pre-select a gene so the visualization opens with expression coloring already visible
- Pre-zoom the spatial view to a region of interest
- Set the initial cell type selection

In [ ]:
import os
from os.path import join, isdir
import scanpy as sc

from vitessce import VitessceConfig, ViewType as vt, AnnDataWrapper
from vitessce.data_utils import VAR_CHUNK_SIZE

In [ ]:
adata = sc.datasets.pbmc68k_reduced()

# Run t-SNE to be able to demonstrate sc.pl.tsne.
sc.tl.tsne(adata)

zarr_path = join("data", "pbmc68k.adata.zarr")
if not isdir(zarr_path):
    os.makedirs("data", exist_ok=True)
    adata.write_zarr(zarr_path, chunks=[adata.shape[0], VAR_CHUNK_SIZE])

adata

In [ ]:
# Build config
vc = VitessceConfig(schema_version="1.0.18", name="Linked Views: Initial Gene Selection")
dataset = vc.add_dataset(name="PBMC").add_object(AnnDataWrapper(
    adata_store=zarr_path,
    obs_set_paths=["obs/bulk_labels"],
    obs_set_names=["Cell Type"],
    obs_embedding_paths=["obsm/X_umap", "obsm/X_tsne"],
    obs_embedding_names=["UMAP", "t-SNE"],
    obs_feature_matrix_path="X",
))

umap     = vc.add_view(vt.SCATTERPLOT, dataset=dataset)
tsne     = vc.add_view(vt.SCATTERPLOT, dataset=dataset)
obs_sets = vc.add_view(vt.OBS_SETS, dataset=dataset)
genes    = vc.add_view(vt.FEATURE_LIST, dataset=dataset)

vc.link_views_by_dict([umap], { "embeddingType": "UMAP" })
vc.link_views_by_dict([tsne], { "embeddingType": "t-SNE" })

# Pre-select CD79A and use gene-expression coloring from the start
vc.link_views_by_dict([umap, genes], {
    "featureSelection": ["CD79A"],
    "obsColorEncoding": "geneSelection",
})

vc.link_views_by_dict([tsne, obs_sets], {
    "obsSetSelection": [["Cell Type", "Dendritic"]],
    "obsSetExpansion": [["Cell Type"]],
    "obsColorEncoding": "cellSetSelection",
})

vc.layout((umap | tsne) / (genes | obs_sets))
vc.widget()

## 3. Independent coordination scopes with `add_coordination()`

Sometimes you want two views to be **independent** — for example, two scatterplots each showing a different gene simultaneously, so you can compare expression patterns side-by-side.

By default, two scatterplots share the same `featureSelection` scope. To give each its own independent scope, use `add_coordination()` to create separate scopes and `view.use_coordination()` to bind each view to its own scope:

```python
[scope_b] = config.add_coordination("featureSelection")
scope_b.set_value(["LYZ"])            # initial value for this scope
umap_b.use_coordination(scope_b)      # bind this view to scope_b only
```

The example below creates two UMAPs: the left one starts on `CD79A` (a B-cell marker), the right one starts on `LYZ` (a monocyte marker). They can be navigated independently.

In [ ]:
vc2 = VitessceConfig(schema_version="1.0.18", name="Independent Gene Selections")
dataset2 = vc2.add_dataset(name="PBMC").add_object(AnnDataWrapper(
    adata_store=zarr_path,
    obs_set_paths=["obs/bulk_labels"],
    obs_set_names=["Cell Type"],
    obs_embedding_paths=["obsm/X_umap"],
    obs_embedding_names=["UMAP"],
    obs_feature_matrix_path="X",
))

umap_a = vc2.add_view(vt.SCATTERPLOT, dataset=dataset2)
umap_b = vc2.add_view(vt.SCATTERPLOT, dataset=dataset2)

vc2.link_views_by_dict([umap_a, umap_b], {
    "embeddingType": "UMAP",
    "obsColorEncoding": "geneSelection",
    "embeddingZoom": None,
    "embeddingTargetX": None,
    "embeddingTargetY": None,
})

# Give each scatterplot its own independent featureSelection scope
[scope_a, scope_b] = vc2.add_coordination("featureSelection", "featureSelection")
scope_a.set_value(["CD79A"])
scope_b.set_value(["LYZ"])

umap_a.use_coordination(scope_a)
umap_b.use_coordination(scope_b)

vc2.layout(umap_a | umap_b)
vc2.widget()

## 4. Advanced coordination with `link_views_by_dict()`

We used `link_views_by_dict()` above with a "flat" dictionary, but this function also supports **nested** coordination via `CoordinationLevel` (CL). This is the mechanism used for image-layer properties like channel colors, which are themselves nested structures (image layer -> image channel -> image channel properties).

The pattern:
```python
from vitessce import CoordinationLevel as CL, get_initial_coordination_scope_prefix

# ...

vc.link_views_by_dict(
    [spatial, layer_controller],
    {
        # Top-level properties
        "imageLayer": CL([
            {
                # Layer-level properties
                "photometricInterpretation": "BlackIsZero",
                "imageChannel": CL([
                    {
                        # Channel-level properties
                        "spatialTargetC": 0,
                        "spatialChannelColor": [255, 0, 0]
                    },
                    {
                        "spatialTargetC": 1,
                        "spatialChannelColor": [0, 255, 0]
                    },
                ]),
            }
        ]),
    },
    scope_prefix=get_initial_coordination_scope_prefix("A", "image"),
)

# ...
```


## 5. Linked spatial + single-cell dashboard

The most powerful use of coordination is linking a **spatial view** and a **scatterplot** so that:

- Selecting a cell type in the OBS_SETS panel highlights those cells in both the tissue image and the UMAP
- Selecting a gene in the FEATURE_LIST colors both the Visium spots and the UMAP points by that gene's expression

We achieve this by building a `SpatialDataWrapper` that exposes both the spatial coordinates and the gene expression matrix, and adding both spatial and non-spatial views to the same configuration.

The Visium dataset used here (breast cancer, 10x Genomics) has ~3,800 spots, each with expression measurements for ~16,000 genes.

In [ ]:
import zipfile
from os.path import isfile, isdir
from urllib.request import urlretrieve

In [ ]:
data_dir = "data"
sdata_filepath = join(data_dir, "visium.spatialdata.zarr")

if not isdir(sdata_filepath):
    zip_filepath = join(data_dir, "visium.spatialdata.zarr.zip")
    if not isfile(zip_filepath):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve(
            "https://data-2.vitessce.io/sdata-datasets/visium.spatialdata.zarr.zip",
            zip_filepath,
        )
    with zipfile.ZipFile(zip_filepath, "r") as z:
        z.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_filepath)

print("Dataset ready.")

## Inspect the contents of the spatialdata object

In [ ]:
from spatialdata import read_zarr 

In [ ]:
sdata = read_zarr(sdata_filepath)
sdata

## Spatial 1

In [ ]:
from vitessce import (
    VitessceConfig,
    ViewType as vt,
    SpatialDataWrapper,
    CoordinationLevel as CL,
    get_initial_coordination_scope_prefix
)

vc1 = VitessceConfig(schema_version="1.0.18", name='Visium SpatialData Demo (visium_associated_xenium_io)')

# Add data to the configuration:
wrapper = SpatialDataWrapper(
    sdata_store=sdata_filepath,
    # The following paths are relative to the root of the SpatialData zarr store on-disk.
    image_path="images/ST8059048_hires_image",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    obs_spots_path="shapes/ST8059048",
    
    coordinate_system="ST8059048",
    coordination_values={
        # The following tells Vitessce to consider each observation as a "spot"
        "obsType": "spot",
    }
)

dataset_uid = "A"
dataset = vc1.add_dataset(name='Breast Cancer Visium', uid=dataset_uid).add_object(wrapper)

# Add views (visualizations) to the configuration:
spatial = vc1.add_view("spatialBeta", dataset=dataset)
layer_controller = vc1.add_view("layerControllerBeta", dataset=dataset)
feature_list = vc1.add_view(vt.FEATURE_LIST, dataset=dataset)

vc1.link_views_by_dict([spatial, layer_controller, feature_list], {
    "obsType": wrapper.obs_type_label,
})

vc1.link_views_by_dict([spatial, layer_controller], {
    'imageLayer': CL([{
        'photometricInterpretation': 'RGB',
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix(dataset_uid, "image"))

vc1.link_views_by_dict([spatial, layer_controller], {
    'spotLayer': CL([{
        'spatialSpotFilled': True,
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix(dataset_uid, "obsSpots"))

# Layout the views
vc1.layout(spatial | (layer_controller / feature_list));

vc1.widget()

## Spatial 2

Pre-select a gene for the spot layer.

In [ ]:
from vitessce import (
    VitessceConfig,
    ViewType as vt,
    SpatialDataWrapper,
    CoordinationLevel as CL,
    get_initial_coordination_scope_prefix
)

vc2 = VitessceConfig(schema_version="1.0.18", name='Visium SpatialData Demo (visium_associated_xenium_io)')

# Add data to the configuration:
wrapper = SpatialDataWrapper(
    sdata_store=sdata_filepath,
    # The following paths are relative to the root of the SpatialData zarr store on-disk.
    image_path="images/ST8059048_hires_image",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    obs_spots_path="shapes/ST8059048",
    
    coordinate_system="ST8059048",
    coordination_values={
        # The following tells Vitessce to consider each observation as a "spot"
        "obsType": "spot",
    }
)

dataset_uid = "A"
dataset = vc2.add_dataset(name='Breast Cancer Visium', uid=dataset_uid).add_object(wrapper)

# Add views (visualizations) to the configuration:
spatial = vc2.add_view("spatialBeta", dataset=dataset)
layer_controller = vc2.add_view("layerControllerBeta", dataset=dataset)
feature_list = vc2.add_view(vt.FEATURE_LIST, dataset=dataset)

# We use .add_coordination to obtain a reference to a "featureSelection" scope as a Python variable.
[gene_selection_scope] = vc2.add_coordination("featureSelection")
gene_selection_scope.set_value(["Sdhb"])

vc2.link_views_by_dict([spatial, layer_controller, feature_list], {
    "obsType": wrapper.obs_type_label,
})

vc2.link_views_by_dict([spatial, layer_controller], {
    'imageLayer': CL([{
        'photometricInterpretation': 'RGB',
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix(dataset_uid, "image"))

# We pass the `gene_selection_scope` variable as the value of `featureSelection` in both .link_views_by_dict calls below.
vc2.link_views_by_dict([spatial, layer_controller], {
    'spotLayer': CL([{
        'spatialSpotFilled': True,
        'featureSelection': gene_selection_scope,
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix(dataset_uid, "obsSpots"))

vc2.link_views_by_dict([feature_list], {
    'featureSelection': gene_selection_scope,
})

# Layout the views
vc2.layout(spatial | (layer_controller / feature_list));

vc2.widget()

## Spatial 3

Side by side Visium samples

In [ ]:
from vitessce import (
    VitessceConfig,
    ViewType as vt,
    SpatialDataWrapper,
    CoordinationLevel as CL,
    get_initial_coordination_scope_prefix
)

vc3 = VitessceConfig(schema_version="1.0.18", name='Visium SpatialData Demo (visium_associated_xenium_io)')

# Add data to the configuration:
wrapper1 = SpatialDataWrapper(
    sdata_store=sdata_filepath,
    # The following paths are relative to the root of the SpatialData zarr store on-disk.
    image_path="images/ST8059048_hires_image",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    obs_spots_path="shapes/ST8059048",
    
    coordinate_system="ST8059048",
    coordination_values={
        # The following tells Vitessce to consider each observation as a "spot"
        "obsType": "spot",
    }
)
wrapper2 = SpatialDataWrapper(
    sdata_store=sdata_filepath,
    # The following paths are relative to the root of the SpatialData zarr store on-disk.
    image_path="images/ST8059050_hires_image",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    obs_spots_path="shapes/ST8059050",
    
    coordinate_system="ST8059050",
    coordination_values={
        # The following tells Vitessce to consider each observation as a "spot"
        "obsType": "spot",
    }
)

dataset1_uid = "ST8059048"
dataset1 = vc3.add_dataset(name='Visium sample 1', uid=dataset1_uid).add_object(wrapper1)

dataset2_uid = "ST8059050"
dataset2 = vc3.add_dataset(name='Visium sample 2', uid=dataset2_uid).add_object(wrapper2)

# Add views (visualizations) to the configuration:
spatial1 = vc3.add_view("spatialBeta", dataset=dataset1)
spatial2 = vc3.add_view("spatialBeta", dataset=dataset2)

layer_controller = vc3.add_view("layerControllerBeta", dataset=dataset1)
feature_list = vc3.add_view(vt.FEATURE_LIST, dataset=dataset1)

# We use .add_coordination to obtain a reference to a "featureSelection" scope as a Python variable.
[gene_selection_scope] = vc3.add_coordination("featureSelection")
gene_selection_scope.set_value(["Sdhb"])

vc3.link_views_by_dict([spatial1, spatial2, layer_controller, feature_list], {
    "obsType": wrapper.obs_type_label,
})

vc3.link_views_by_dict([spatial1, layer_controller], {
    'imageLayer': CL([{
        'photometricInterpretation': 'RGB',
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix(dataset1_uid, "image"))

vc3.link_views_by_dict([spatial2], {
    'imageLayer': CL([{
        'photometricInterpretation': 'RGB',
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix(dataset2_uid, "image"))

# We pass the `gene_selection_scope` variable as the value of `featureSelection` in both .link_views_by_dict calls below.
vc3.link_views_by_dict([spatial1, layer_controller], {
    'spotLayer': CL([{
        'spatialSpotFilled': True,
        'featureSelection': gene_selection_scope,
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix(dataset1_uid, "obsSpots"))

vc3.link_views_by_dict([spatial2], {
    'spotLayer': CL([{
        'spatialSpotFilled': True,
        'featureSelection': gene_selection_scope,
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix(dataset2_uid, "obsSpots"))

vc3.link_views_by_dict([feature_list], {
    'featureSelection': gene_selection_scope,
})

# Layout the views
vc3.layout((spatial1 | spatial2) / (layer_controller | feature_list));

vc3.widget()

## Exercises:
- Link the zoom levels of the two spatial views.
- Link spot-layer properties of both spatial views (opacity, filled/stroked, stroke width, etc.)